## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:吕佩哲


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <bits/stdc++.h>
using namespace std;

static const int LIM = 1024 + 5;

int m;                  // 实际上是 2^k
int s1, s2;             // 固定交换的两个数字
int blockSize;          // lowbitLen

int arr[LIM], wherePos[LIM];

vector<pair<int,int>> ops; 
// {0,0} -> swap
// {1,x} -> xor x
// {2,x} -> add x

inline int fastRead() {
    int v = 0, ch = getchar();
    while (ch < '0' || ch > '9') ch = getchar();
    while (ch >= '0' && ch <= '9') {
        v = v * 10 + (ch - '0');
        ch = getchar();
    }
    return v;
}

inline void refreshPos() {
    for (int i = 0; i < m; ++i) {
        wherePos[arr[i]] = i;
    }
}

inline void doSwapMagic() {
    ops.push_back({0, 0});
    for (int i = 0; i < m; ++i) {
        if (arr[i] == s1) arr[i] = s2;
        else if (arr[i] == s2) arr[i] = s1;
    }
    refreshPos();
}

inline void doAddMagic(int delta) {
    delta %= m;
    if (delta < 0) delta += m;
    if (delta == 0) return;
    ops.push_back({2, delta});
    for (int i = 0; i < m; ++i) {
        arr[i] = (arr[i] + delta) % m;
    }
    refreshPos();
}

inline void doXorMagic(int mask) {
    if (mask == 0) return;
    ops.push_back({1, mask});
    for (int i = 0; i < m; ++i) {
        arr[i] ^= mask;
    }
    refreshPos();
}

inline void calcBridgePos(int x, int y, int &px, int &py) {
    int diff = (y - x + m - blockSize + m) % m;
    px = py = 0;
    for (int step = m / 2; step >= 2 * blockSize; step >>= 1) {
        if (diff >= step) {
            diff -= step;
            py += step / 2;
        } else {
            px += step / 2;
        }
    }
    int tail = (x & (blockSize - 1));
    px += m / 2 + tail;
    py += tail;
}

void simulateSwapPair(int x, int y) {
    int gx = (x / blockSize) & 1;
    int gy = (y / blockSize) & 1;

    if (gx == gy) {
        int mid;
        if (gx == 0) mid = (x & (blockSize - 1)) + blockSize;
        else mid = (x & (blockSize - 1));
        simulateSwapPair(x, mid);
        simulateSwapPair(y, mid);
        simulateSwapPair(x, mid);
        return;
    }

    int ps1, ps2, px, py;
    calcBridgePos(s1, s2, ps1, ps2);
    calcBridgePos(x, y, px, py);

    doAddMagic((px - x + m) % m);
    doXorMagic(px ^ ps1);
    doAddMagic((s1 - ps1 + m) % m);

    doSwapMagic();

    doAddMagic((ps1 - s1 + m) % m);
    doXorMagic(px ^ ps1);
    doAddMagic((x - px + m) % m);
}

struct Builder {
    int len;
    int p[LIM];
    vector<int> seq; // 正数表示 add，负数表示 xor

    bool isSorted() const {
        for (int i = 1; i < len; ++i) {
            if (p[i - 1] > p[i]) return false;
        }
        return true;
    }

    Builder inversePerm() const {
        Builder res;
        res.len = len;
        for (int i = 0; i < len; ++i) {
            res.p[p[i]] = i;
        }
        return res;
    }

    bool construct() {
        vector<int> seen(len, 0);
        for (int i = 0; i < len; ++i) {
            if (p[i] < 0 || p[i] >= len) return false;
            seen[p[i]] = 1;
        }
        for (int i = 0; i < len; ++i) {
            if (!seen[i]) return false;
        }

        if (len == 1) return true;

        Builder leftHalf, rightHalf;
        leftHalf.len = rightHalf.len = len / 2;

        for (int i = 0; i < len / 2; ++i) {
            leftHalf.p[i] = p[i * 2] >> 1;
            rightHalf.p[i] = p[i * 2 + 1] >> 1;
        }

        if (!leftHalf.construct()) return false;
        if (!rightHalf.construct()) return false;

        if (p[0] & 1) {
            seq.push_back(len == 2 ? 1 : -1);
        }

        int tagL = 0;
        for (int v : leftHalf.seq) {
            if (v > 0) {
                seq.push_back(-1);
                seq.push_back(1);
            } else {
                seq.push_back(v * 2);
                tagL ^= ((-v) * 2);
            }
        }
        if (tagL) seq.push_back(-tagL);

        int tagR = 0;
        for (int v : rightHalf.seq) {
            if (v > 0) {
                seq.push_back(1);
                seq.push_back(-1);
            } else {
                seq.push_back(v * 2);
                tagR ^= ((-v) * 2);
            }
        }

        if ((tagL & (len / 2)) != (tagR & (len / 2))) return false;

        if (tagL >= len / 2) tagL -= len / 2;
        if (tagR >= len / 2) tagR -= len / 2;
        if (tagL != tagR) return false;

        vector<int> compressed;
        for (int v : seq) {
            if (compressed.empty()) {
                compressed.push_back(v);
            } else if (v < 0 && compressed.back() < 0) {
                int nv = ((-compressed.back()) ^ (-v));
                compressed.pop_back();
                if (nv) compressed.push_back(-nv);
            } else {
                compressed.push_back(v);
            }
        }
        seq.swap(compressed);
        return true;
    }
};

int main() {
    m = fastRead();
    s1 = fastRead();
    s2 = fastRead();

    for (int i = 0; i < m; ++i) arr[i] = fastRead();
    refreshPos();

    int d = (s1 - s2 + m) % m;
    blockSize = d & (-d);
    if (blockSize == 0) blockSize = m;

    if (blockSize > 1) {
        Builder solver;
        solver.len = blockSize;
        for (int i = 0; i < m; ++i) {
            solver.p[i] = arr[i] & (blockSize - 1);
        }

        if (!solver.construct()) {
            puts("-1");
            return 0;
        }

        for (int v : solver.seq) {
            if (v > 0) doAddMagic(v);
            else doXorMagic(-v);
        }
    }

    for (int r = 0; r < blockSize; ++r) {
        vector<int> bucket;
        for (int i = r; i < m; i += blockSize) {
            bucket.push_back(arr[i]);
        }

        sort(bucket.begin(), bucket.end());

        bool feasible = true;
        for (int i = 0, pos = r; pos < m; pos += blockSize, ++i) {
            if (bucket[i] != pos) {
                feasible = false;
                break;
            }
        }

        if (!feasible) {
            puts("-1");
            return 0;
        }

        for (int pos = r; pos < m; pos += blockSize) {
            if (arr[pos] != pos) {
                simulateSwapPair(pos, arr[pos]);
            }
        }
    }

    for (int i = 0; i < m; ++i) {
        assert(arr[i] == i);
    }

    printf("%d\n", (int)ops.size());
    for (auto &e : ops) {
        if (e.first == 0) {
            printf("0\n");
        } else {
            printf("%d %d\n", e.first, e.second);
        }
    }
    return 0;
}

## B 长跑

In [ ]:
#include <bits/stdc++.h>
using namespace std;

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int stationCount, totalDistance, maxRunDistance, budget;
    while (cin >> stationCount >> totalDistance >> maxRunDistance >> budget) {
        // 同一位置可能有多个补给站，只保留该位置最便宜的那个
        map<int, int> cheapestAtPos;
        for (int i = 0; i < stationCount; ++i) {
            int position, price;
            cin >> position >> price;
            if (!cheapestAtPos.count(position)) cheapestAtPos[position] = price;
            else cheapestAtPos[position] = min(cheapestAtPos[position], price);
        }

        // 起点就已经在终点
        if (totalDistance == 0) {
            cout << "Yes\n";
            continue;
        }

        // 连起点到终点都跑不到，而且体力为 0，则一定不行
        if (maxRunDistance == 0) {
            cout << "No\n";
            continue;
        }

        vector<pair<int, int>> supplyPoints;
        for (auto &entry : cheapestAtPos) {
            // 超出终点右侧的补给点没意义；起点左侧也不可能
            if (entry.first >= 0 && entry.first <= totalDistance) {
                supplyPoints.push_back(entry);
            }
        }

        int pointCount = (int)supplyPoints.size();
        const int INF = 1e9;
        vector<int> minCostToReach(pointCount, INF);
        // minCostToReach[i]：到达第 i 个补给点并在这里补满体力的最小花费

        for (int i = 0; i < pointCount; ++i) {
            int currPos = supplyPoints[i].first;
            int replenishCost = supplyPoints[i].second;

            // 从起点直接跑到这个补给点
            if (currPos <= maxRunDistance) {
                minCostToReach[i] = replenishCost;
            }

            // 从前面的某个补给点补满后跑到这里
            for (int j = 0; j < i; ++j) {
                int prevPos = supplyPoints[j].first;
                if (currPos - prevPos <= maxRunDistance && minCostToReach[j] != INF) {
                    minCostToReach[i] = min(minCostToReach[i], minCostToReach[j] + replenishCost);
                }
            }
        }

        bool canReachEnd = false;

        // 不补给，直接到终点
        if (totalDistance <= maxRunDistance) canReachEnd = true;

        // 或者在某个补给点补给后到终点
        for (int i = 0; i < pointCount && !canReachEnd; ++i) {
            int currPos = supplyPoints[i].first;
            if (totalDistance - currPos <= maxRunDistance && minCostToReach[i] <= budget) {
                canReachEnd = true;
            }
        }

        cout << (canReachEnd ? "Yes" : "No") << '\n';
    }

    return 0;
}

## C 最长回文

In [ ]:
#include<bits/stdc++.h>
using namespace std;
const int N = 3e5 + 10;
string s1 , s2;  
int p1[N] , p2[N] , global_max = 1; 
string Manacher(string str , int *p) 
{
    string t = "$#";
    for(auto ch : str) t += ch , t += '#';  
    int mx = 0 , id = 0 ;
    int len = t.size() , ans = 0; 
    for(int i = 1 ; i < len ; i ++)
    {
        p[i] = mx > i ? min(p[2 * id - i] , mx - i) : 1;
        while(t[i + p[i]] == t[i - p[i]]) p[i] ++ ;
        if(mx < i + p[i]) mx = i + p[i] , id = i;
        ans = max(ans , p[i] - 1);
    }
    global_max = max(global_max , ans); 
    return t;
}
signed main()
{
    int n ;
    cin >> n >> s1 >> s2;  
    s1 = Manacher(s1 , p1);  
    s2 = Manacher(s2 , p2);  
    n = n * 2 + 2;
    int ans = 1;
    for(int i = 2 ; i <= n ; i ++)
    {
        int len = max(p1[i] , p2[i - 2]);  
        while(s1[i - len] == s2[i - 2 + len]) len ++; 
        ans = max(ans , len - 1); 
    }
    cout << ans << '\n';
    return 0;
}

## D 优惠券

In [ ]:
#include<iostream>
#include<string>
#include<map>
#include<set>
#include<algorithm>
using namespace std;
map<int,int> mp;      // 记录每个值对应的最近操作时间（正为插入，负为删除）
set<int> ques;        // 存储所有问号操作的位置

int main()
{
    int m;
    string op;        // 操作类型：I（插入）、D（删除）、?（查询）
    int x;            // 操作的值
    while(cin>>m)     // 处理多组测试数据
    {
        mp.clear();
        ques.clear();
        set<int>::iterator it;
        int ans=-1;   // 记录第一个导致错误的操作编号，-1表示没有错误
        bool flag=true; // 标记当前是否还有效（是否已经出现过错误）
        for(int i=1;i<=m;i++)  // i为当前操作的编号（从1开始）
        {
            cin>>op;
            if(op=="?")   // 问号操作：记录当前操作位置
            {
                ques.insert(i);
            }
            else          // 插入或删除操作
            {
                cin>>x;
                if(!flag) continue; // 已经发生错误，后续操作不再处理
                if(op=="I")   // 插入操作
                {
                    if(mp[x]>0)   // 该值之前已插入未删除，需先匹配掉一个问号
                    {
                        it=ques.lower_bound(mp[x]); // 找第一个大于等于上次操作位置的问号
                        if(it==ques.end()) ans=i,flag=false; // 没有合适的问号匹配，错误
                        else ques.erase(it);                // 匹配成功，移除该问号
                    }
                    mp[x]=i; // 记录当前插入操作的位置
                }
                else        // 删除操作（op=="D"）
                {
                    if(mp[x]<=0)  // 该值之前处于删除状态或不存在，需匹配一个问号
                    {
                        it=ques.lower_bound(-mp[x]); // 找第一个大于等于上次操作绝对值的问号
                        if(it==ques.end()) ans=i,flag=false; // 没有问号可匹配，错误
                        else ques.erase(it);                // 匹配成功
                    }
                    mp[x]=-i; // 记录删除操作的位置（用负数标记）
                }
            }
        }
        cout<<ans<<endl; // 输出第一个错误的位置，若无错误则输出-1
    }
    return 0;
}

## E 任意点

In [ ]:
#include <iostream>
#include <map>
using namespace std;
#define IOS ios::sync_with_stdio(false); cin.tie(0); cout.tie(0);
// 加快输入输出速度

const int MAX = 1e2 + 7;
int n;              // 点的数量
int parent[MAX];    // 并查集父节点数组
pair<int, int> points[MAX];  // 存储每个点的坐标 (x, y)

int findRoot(int x);    // 查找根节点（路径压缩）
void unionSet(int x, int y);  // 合并两个点所在的集合

int main() {
    IOS;
    cin >> n;
    // 初始化并查集，每个点自成一个集合
    for (int i = 1; i <= n; i++) {
        parent[i] = i;
    }
    // 读入所有点的坐标
    for (int i = 1; i <= n; i++) {
        cin >> points[i].first >> points[i].second;
    }
    // 如果两个点有相同的 x 坐标或相同的 y 坐标，则它们属于同一连通块
    for (int i = 1; i <= n; i++) {
        for (int j = i + 1; j <= n; j++) {
            if (points[i].first == points[j].first || points[i].second == points[j].second)
                unionSet(i, j);
        }
    }
    // 统计有多少个不同的连通块
    int ans = 0;
    for (int i = 1; i <= n; i++) {
        if (i == findRoot(i)) {
            ans++;
        }
    }
    // 最少需要添加的线段数量 = 连通块数 - 1
    cout << ans - 1 << endl;
    return 0;
}

// 查找 x 所在集合的根节点，带路径压缩优化
int findRoot(int x) {
    return x == parent[x] ? x : parent[x] = findRoot(parent[x]);
}

// 合并 x 和 y 所在的集合
void unionSet(int x, int y) {
    x = findRoot(x);
    y = findRoot(y);
    if (x != y) {
        parent[x] = y;
    }
}

## F 通配符匹配

In [ ]:
#include<iostream>
#include<cstdio>
#include<cstring>
#include<algorithm>
#define ll unsigned long long
ll base = 19260817ll;  // 哈希基数
using namespace std;
ll pre[100010], mul[100010], seg_hash[20];  // 前缀哈希、基数幂、模式片段哈希
int n, seg_cnt;        // 文本串数量、模式片段数量
int has_star[20];      // 标记片段前是否有'*'（1表示有*，0表示只有?或没有）
int seg_len[20];       // 每个模式片段的长度
int dp[15][100010];    // DP数组：dp[i][j]表示匹配到第i个片段、文本第j个字符时的状态
char seg_str[15][100010], text[100010], temp[100010];  // 模式片段、文本串、临时数组

// 计算字符串s的前len个字符的哈希值
ll get_hash(char s[], int len) {
    ll res = 0;
    for (int i = 0; i < len; i++) 
        res *= base, res += (ll)s[i];
    return res;
}

int main() {
    scanf("%s", text);  // 读入模式串（包含通配符*和?）
    int i, j, k, l, len = strlen(text);
    ll t1;  // 临时变量，用于比较哈希值
    
    // 预处理base的幂
    mul[0] = 1;
    for (i = 1; i <= 100000; i++) 
        mul[i] = mul[i-1] * base;
    
    // 处理模式串开头的通配符
    if (text[0] == '*' || text[0] == '?') 
        seg_hash[++seg_cnt] = get_hash(seg_str[seg_cnt], seg_len[seg_cnt] = 0);
    
    // 将模式串按通配符分割成若干普通字符串片段
    for (i = 0; i < len; ) {
        if (text[i] == '*' || text[i] == '?') {
            has_star[seg_cnt] = (has_star[seg_cnt] || (text[i] == '*'));  // 记录是否有*
            i++;
        }
        j = 0;
        seg_cnt++;
        // 提取连续的非通配符字符作为一个片段
        while (text[i] != '*' && text[i] != '?' && i < len) 
            seg_str[seg_cnt][j] = text[i], i++, j++;
        seg_hash[seg_cnt] = get_hash(seg_str[seg_cnt], seg_len[seg_cnt] = j);
    }
    
    len++;
    // 处理模式串结尾的通配符
    if (text[len-2] == '*' || text[len-2] == '?') 
        seg_cnt++, has_star[seg_cnt] = seg_len[seg_cnt] = seg_hash[seg_cnt] = 0;
    else 
        has_star[seg_cnt] = 0;
    
    scanf("%d", &n);  // 待匹配的文本串数量
    for (l = 1; l <= n; l++) {
        memset(text, 0, sizeof(text));
        scanf("%s", text);
        len = strlen(text);
        text[len] = '$';  // 添加结束标记
        len++;
        
        // 初始化DP数组
        memset(dp, -1, sizeof(dp));
        dp[0][0] = 2;  // 初始状态
        pre[0] = 0;
        
        // 计算文本串的前缀哈希，便于O(1)获取子串哈希
        for (i = 0; i < len; i++) 
            pre[i+1] = pre[i] * base + (ll)text[i];
        
        // DP匹配过程
        for (j = 0; j <= len; j++) {
            for (i = 0; i <= seg_cnt; i++) {
                if (dp[i][j] == -1) continue;
                
                // 状态1：当前可以匹配任意字符（由*产生）
                if (dp[i][j] == 1) 
                    dp[i][j+1] = 1;
                
                // 状态0或2：处理空匹配或普通字符匹配
                if (!dp[i][j]) {
                    dp[i][j] = 2;
                    if (dp[i][j+1] == -1) dp[i][j+1] = 2;
                    if (dp[i][j+1] == 0) dp[i][j+1] = 3;
                    continue;
                }
                
                // 状态3处理
                if (dp[i][j] == 3) {
                    dp[i][j] = 2;
                    if (dp[i][j+1] == -1) dp[i][j+1] = 2;
                    if (dp[i][j+1] == 0) dp[i][j+1] = 3;
                }
                
                // 尝试匹配下一个模式片段
                t1 = pre[j + seg_len[i+1]] - pre[j] * mul[seg_len[i+1]];
                if (t1 == seg_hash[i+1]) {  // 哈希匹配成功
                    dp[i+1][j + seg_len[i+1]] = max(dp[i+1][j + seg_len[i+1]], has_star[i+1]);
                }
            }
        }
        
        // 输出匹配结果
        if (dp[seg_cnt][len] != -1) puts("YES");
        else puts("NO");
    }
}

## G 汉诺塔

In [ ]:
#include<iostream>
using namespace std;
long long f[210][7];   // f[i][j] 表示 i 个盘子的汉诺塔，将 j 柱上的盘子全部移动到目标柱的最小步数
long long nxt[210][7]; // nxt[i][j] 表示 i 个盘子的汉诺塔，将 j 柱上的盘子全部移动后，目标柱的编号
bool vis[7];           // 标记柱子是否已在输入中出现
char rule[10];         // 存储输入的移动规则

int main() {
    int n;
    cin >> n;
    
    // 读入 6 条移动规则（每个规则表示从某柱移动到某柱）
    for (int i = 1; i <= 6; i++) {
        scanf("%s", rule);
        int from = rule[0] - 64;  // 将 A/B/C 转换为 1/2/3
        int to = rule[1] - 64;
        if (vis[from]) continue;   // 每个起始柱只取第一条规则
        vis[from] = 1;
        nxt[1][from] = to;         // 1 个盘子时，从 from 移动到 to
        f[1][from] = 1;            // 1 个盘子只需要 1 步
    }
    
    // 递推计算更多盘子的情况
    for (int i = 2; i <= n; i++) {
        for (int j = 1; j <= 3; j++) {
            int a = j;                     // 起始柱
            int b = nxt[i-1][j];           // 中间柱（i-1 个盘子移动的目标）
            int c = 1 + 2 + 3 - a - b;     // 目标柱（剩余的那一根）
            
            // 情况1：最优解符合经典汉诺塔策略
            if (nxt[i-1][b] == c) {
                f[i][a] = f[i-1][a] + f[i-1][b] + 1;
                nxt[i][a] = c;
            }
            // 情况2：最优解需要额外搬运
            if (nxt[i-1][b] == a) {
                f[i][a] = f[i-1][a] + 1 + f[i-1][b] + 1 + f[i-1][a];
                nxt[i][a] = b;
            }
        }
    }
    
    cout << f[n][1] << endl;  // 输出将 n 个盘子从 A 柱移动到目标柱的最小步数
    return 0;
}

## H 马步距离

In [ ]:
#include<bits/stdc++.h>
using namespace std;

int step;           // 记录最小步数
int start_x, start_y, target_x, target_y;  // 起点和终点坐标
int knight_moves[5][5] = {  // 小范围（0~4）内骑士从(0,0)到(x,y)的预计算步数表
    {0, 3, 2, 3, 2},
    {3, 2, 1, 2, 3},
    {2, 1, 4, 3, 2},
    {3, 2, 3, 2, 3},
    {2, 3, 2, 3, 4}
};

int main() {
    scanf("%d%d%d%d", &start_x, &start_y, &target_x, &target_y);
    int dx = abs(start_x - target_x);  // x方向上的距离
    int dy = abs(start_y - target_y);  // y方向上的距离
    
    // 当距离较大时，使用贪心策略快速逼近
    while (dx > 4 || dy > 4) {
        if (dx < 0) dx = -dx;
        if (dy < 0) dy = -dy;
        if (dx < dy) swap(dx, dy);  // 保证 dx >= dy
        dx -= 2;
        dy -= 1;
        step++;
    }
    
    // 剩余小范围距离查表得到精确步数
    printf("%d\n", step + knight_moves[dx][dy]);
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        // 单调栈解法：找到每个柱子左右两边第一个比它矮的位置
        int n = heights.size();
        if (n == 0) return 0;
        
        stack<int> st;  // 单调递增栈，存储柱子下标
        int max_area = 0;
        
        for (int i = 0; i <= n; i++) {
            // 在数组末尾加一个高度为0的哨兵，确保最后能清空栈
            int cur_height = (i == n) ? 0 : heights[i];
            
            // 当当前高度小于栈顶高度时，可以计算以栈顶高度为高的矩形面积
            while (!st.empty() && cur_height < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();
                // 确定矩形的左边界
                int left = st.empty() ? -1 : st.top();
                // 宽度为 (i - left - 1)
                int width = i - left - 1;
                max_area = max(max_area, h * width);
            }
            st.push(i);
        }
        
        return max_area;
    }
};

## J 消防局的设立

In [ ]:
#include<iostream>
#include<algorithm>
using namespace std;
#define maxN 1010
#define INF 2000000000

int N;
struct edge {
    int to;
    int next;
} children[maxN];  // 前向星存储子节点
int head[maxN] = {0};  // 前向星头指针
int dp[maxN][5];  // dp[u][k]：以u为根的子树，状态不超过k时的最小代价

int edge_cnt = 0;  // 当前边数量
void addChild(int parent, int child) {
    edge_cnt++;
    children[edge_cnt].to = child;
    children[edge_cnt].next = head[parent];
    head[parent] = edge_cnt;
}

void dfs(int node) {
    // 初始化状态
    dp[node][0] = 1;   // 状态0：节点放监控（代价1）
    dp[node][3] = 0;   // 状态3：节点被父节点覆盖（代价0）
    dp[node][4] = 0;   // 状态4：节点被某后代覆盖（代价0）
    
    // 先递归处理所有子节点
    for (int i = head[node]; i; i = children[i].next) {
        int child = children[i].to;
        dfs(child);
        dp[node][0] += dp[child][4];      // 节点放监控，子节点可被孙子覆盖
        dp[node][3] += dp[child][2];      // 节点被父覆盖，子节点需被自己或孙子覆盖
        dp[node][4] += dp[child][3];      // 节点被后代覆盖，子节点可被父覆盖
    }
    
    // 处理叶子节点特殊情况
    if (head[node] == 0) {
        dp[node][1] = dp[node][2] = 1;    // 叶子节点需放监控或被子节点（不存在）覆盖
    }
    else {
        dp[node][1] = dp[node][2] = INF;
        // 枚举哪个子节点放置监控来覆盖当前节点
        for (int i = head[node]; i; i = children[i].next) {
            int child1 = children[i].to;
            int sum1 = dp[child1][0];   // 该子节点放监控
            int sum2 = dp[child1][1];   // 该子节点被自己子节点覆盖
            // 其他子节点取最小可行状态
            for (int j = head[node]; j; j = children[j].next) {
                if (i == j) continue;
                int child2 = children[j].to;
                sum1 += dp[child2][3];  // 其他子节点由父节点覆盖
                sum2 += dp[child2][2];  // 其他子节点由自己或孙子覆盖
            }
            dp[node][1] = min(dp[node][1], sum1);  // 状态1：节点被子节点覆盖
            dp[node][2] = min(dp[node][2], sum2);  // 状态2：节点被孙子覆盖
        }
    }
    
    // 状态取前缀最小值，保证单调性
    for (int i = 1; i <= 4; i++) {
        dp[node][i] = min(dp[node][i], dp[node][i-1]);
    }
}

int main() {
    cin >> N;
    for (int i = 2; i <= N; i++) {
        int parent;
        cin >> parent;
        addChild(parent, i);
    }
    dfs(1);
    
    // 输出根节点被自己或孙子覆盖的最小代价
    cout << dp[1][2];
}